# 02. Exploratory Data Analysis & Churn Driver Investigation
**Project:** Telco Customer Churn & Retention Intelligence  
**Phase:** Phase 2 - Exploratory Data Analysis & Customer Behavior Analysis  
**Dataset:** `data/processed/telco_churn_cleaned.csv`

---

## Executive Overview & Analytical Objectives
In this notebook, we investigate the multi-dimensional relationships between customer attributes and customer churn. We explore:
1. **Contractual Commitments:** Month-to-month vs. Annual vs. Two-year plans.
2. **Tenure Dynamics:** Early-tenure hazard vs. long-term account stabilization.
3. **Billing & Payment Methods:** Electronic checks vs. Automated payment rails.
4. **Internet Infrastructure:** Fiber optic vs. DSL vs. Non-internet accounts.
5. **Pricing Sensitivity:** Recurring monthly charges and cumulative spend.
6. **Demographics:** Age, partnership status, and household dependents.
7. **Value-Added Protection Services:** Tech support, online security, and cloud backup.


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.autolayout'] = True

DATA_PATH = (Path.cwd().parent / "data/processed/telco_churn_cleaned.csv" 
             if Path.cwd().name == "notebooks" 
             else Path("data/processed/telco_churn_cleaned.csv")).resolve()
FIGURES_DIR = (Path.cwd().parent / "reports/figures" 
               if Path.cwd().name == "notebooks" 
               else Path("reports/figures")).resolve()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Data successfully loaded. Records: {len(df):,}")


Data successfully loaded. Records: 7,043


## 1. Churn by Contract Type & Internet Service
Contract structure represents the primary operational barrier to exit. We evaluate churn rates across contract types and examine the intersection with internet service tiers.


In [2]:
# Contract Churn Analysis
contract_churn = df.groupby('Contract')['Churn'].agg(
    total_customers='count',
    churned_customers=lambda x: (x == 'Yes').sum(),
    churn_rate_pct=lambda x: (x == 'Yes').mean() * 100
).reset_index()

print("--- Churn by Contract Type ---")
print(contract_churn.to_string(index=False))

# Internet Service Churn Analysis
internet_churn = df.groupby('InternetService')['Churn'].agg(
    total_customers='count',
    churned_customers=lambda x: (x == 'Yes').sum(),
    churn_rate_pct=lambda x: (x == 'Yes').mean() * 100
).reset_index()

print(); print("--- Churn by Internet Service ---")
print(internet_churn.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Contract Churn Rate
bars1 = axes[0].bar(contract_churn['Contract'], contract_churn['churn_rate_pct'],
                    color=['#d9534f', '#f0ad4e', '#5cb85c'], edgecolor='black', width=0.5)
axes[0].set_title("Churn Rate by Contract Commitment", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Churn Rate (%)", fontsize=11)
axes[0].set_ylim(0, 50)
for bar in bars1:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., h + 1, f"{h:.1f}%", ha='center', va='bottom', fontweight='bold')

# Plot 2: Internet Service Churn Rate
bars2 = axes[1].bar(internet_churn['InternetService'], internet_churn['churn_rate_pct'],
                    color=['#f0ad4e', '#d9534f', '#5cb85c'], edgecolor='black', width=0.5)
axes[1].set_title("Churn Rate by Internet Service Infrastructure", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Churn Rate (%)", fontsize=11)
axes[1].set_ylim(0, 50)
for bar in bars2:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 1, f"{h:.1f}%", ha='center', va='bottom', fontweight='bold')

fig1_path = FIGURES_DIR / "02_churn_by_contract_and_internet.png"
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {fig1_path.resolve()}")


--- Churn by Contract Type ---
      Contract  total_customers  churned_customers  churn_rate_pct
Month-to-month             3875               1655       42.709677
      One year             1473                166       11.269518
      Two year             1695                 48        2.831858

--- Churn by Internet Service ---
InternetService  total_customers  churned_customers  churn_rate_pct
            DSL             2421                459       18.959108
    Fiber optic             3096               1297       41.892765
             No             1526                113        7.404980
[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\02_churn_by_contract_and_internet.png


## 2. Tenure Dynamics: The Early-Tenure Churn Zone
We inspect how tenure correlates with customer churn. Telecommunications churn is rarely uniform across the lifecycle; high attrition is typically concentrated in early customer onboarding months.


In [3]:
# Define tenure bands
df['tenure_cohort'] = pd.cut(
    df['tenure'],
    bins=[-1, 6, 12, 24, 36, 48, 60, 72],
    labels=['0-6 mos', '7-12 mos', '13-24 mos', '25-36 mos', '37-48 mos', '49-60 mos', '61-72 mos']
)

cohort_churn = df.groupby('tenure_cohort', observed=False)['Churn'].agg(
    total='count',
    churned=lambda x: (x == 'Yes').sum(),
    churn_rate=lambda x: (x == 'Yes').mean() * 100
).reset_index()

print("--- Churn by Tenure Cohort ---")
print(cohort_churn.to_string(index=False))

# Visualizing Tenure KDE and Cohort Churn Rates
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Tenure KDE Distribution by Churn Status
sns.kdeplot(data=df, x='tenure', hue='Churn', common_norm=False, fill=True, palette={'No': '#2b5c8f', 'Yes': '#d9534f'}, ax=axes[0], alpha=0.4)
axes[0].set_title("Tenure Density: Churned vs. Retained Customers", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Account Tenure (Months)", fontsize=11)
axes[0].set_ylabel("Probability Density", fontsize=11)

# Plot 2: Churn Rate across Tenure Cohorts
bars = axes[1].bar(cohort_churn['tenure_cohort'].astype(str), cohort_churn['churn_rate'], color='#d9534f', edgecolor='black', width=0.55)
axes[1].set_title("Churn Rate by Tenure Cohort (Inflection Analysis)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Tenure Cohort", fontsize=11)
axes[1].set_ylabel("Churn Rate (%)", fontsize=11)
axes[1].set_ylim(0, 60)
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 1, f"{h:.1f}%", ha='center', va='bottom', fontweight='bold')

fig2_path = FIGURES_DIR / "02_tenure_and_cohort_churn.png"
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {fig2_path.resolve()}")


--- Churn by Tenure Cohort ---
tenure_cohort  total  churned  churn_rate
      0-6 mos   1481      784   52.937205
     7-12 mos    705      253   35.886525
    13-24 mos   1024      294   28.710938
    25-36 mos    832      180   21.634615
    37-48 mos    762      145   19.028871
    49-60 mos    832      120   14.423077
    61-72 mos   1407       93    6.609808
[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\02_tenure_and_cohort_churn.png


## 3. Financial Metrics: Monthly Charges & Total Charges
Does price sensitivity drive churn? We compare the recurring `MonthlyCharges` and cumulative `TotalCharges` between churned and retained customer accounts.


In [4]:
# Financial Summary Statistics by Churn
fin_summary = df.groupby('Churn')[['MonthlyCharges', 'TotalCharges']].agg(['mean', 'median', 'std']).round(2)
print("--- Financial Comparison by Churn Status ---")
print(fin_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Monthly Charges Boxplot
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette={'No': '#2b5c8f', 'Yes': '#d9534f'}, ax=axes[0], width=0.4)
axes[0].set_title("Monthly Recurring Charges (MRR) by Churn Status", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Monthly Charges (USD)", fontsize=11)

# Plot 2: Total Charges Boxplot
sns.boxplot(data=df, x='Churn', y='TotalCharges', palette={'No': '#2b5c8f', 'Yes': '#d9534f'}, ax=axes[1], width=0.4)
axes[1].set_title("Cumulative Lifetime Charges by Churn Status", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Total Charges (USD)", fontsize=11)

fig3_path = FIGURES_DIR / "02_charges_boxplots.png"
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {fig3_path.resolve()}")


--- Financial Comparison by Churn Status ---
      MonthlyCharges               TotalCharges                  
                mean median    std         mean   median      std
Churn                                                            
No             61.27  64.43  31.09      2549.91  1679.52  2329.95
Yes            74.44  79.65  24.67      1531.80   703.55  1890.82
[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\02_charges_boxplots.png


<string>:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

<string>:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.



### Analytical Decision & Interpretation Note on TotalCharges
Notice that while churned customers pay significantly **higher** monthly charges (median ~$80 vs. ~$64), their cumulative `TotalCharges` are markedly **lower** (median ~$700 vs. ~$1,680).  
**Confounding Mechanism:** This occurs because churners have dramatically shorter tenure (median 10 months vs. 38 months). Total charges reflect tenure accumulation rather than price affordability.


## 4. Payment Method & Billing Mechanics
We investigate how payment channels and billing friction impact churn.


In [5]:
pay_churn = df.groupby('PaymentMethod')['Churn'].agg(
    total='count',
    churned=lambda x: (x == 'Yes').sum(),
    churn_rate=lambda x: (x == 'Yes').mean() * 100
).reset_index().sort_values(by='churn_rate', ascending=False)

print("--- Churn by Payment Method ---")
print(pay_churn.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(pay_churn['PaymentMethod'], pay_churn['churn_rate'], color=['#d9534f', '#f0ad4e', '#2b5c8f', '#2b5c8f'], edgecolor='black', height=0.5)
ax.set_title("Customer Churn Rate by Payment Channel", fontsize=13, fontweight='bold')
ax.set_xlabel("Churn Rate (%)", fontsize=11)
ax.set_xlim(0, 55)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 1, bar.get_y() + bar.get_height()/2., f"{w:.1f}%", ha='left', va='center', fontweight='bold')

fig4_path = FIGURES_DIR / "02_churn_by_payment_channel.png"
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {fig4_path.resolve()}")


--- Churn by Payment Method ---
            PaymentMethod  total  churned  churn_rate
         Electronic check   2365     1071   45.285412
             Mailed check   1612      308   19.106700
Bank transfer (automatic)   1544      258   16.709845
  Credit card (automatic)   1522      232   15.243101
[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\02_churn_by_payment_channel.png


## 5. Value-Added Services & Customer Retention
Do add-on services act as retention anchors? We examine churn rates for accounts with vs. without technical and security add-ons (`TechSupport`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`).


In [6]:
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

service_records = []
for s in services:
    grouped = df.groupby(s)['Churn'].agg(
        total='count',
        churn_rate=lambda x: (x == 'Yes').mean() * 100
    )
    for category in grouped.index:
        service_records.append({
            "Service": s,
            "Subscription Status": category,
            "Customer Count": grouped.loc[category, 'total'],
            "Churn Rate (%)": round(grouped.loc[category, 'churn_rate'], 1)
        })

service_df = pd.DataFrame(service_records)
print("--- Value-Added Services Churn Summary ---")
print(service_df.to_string(index=False))

# Plotting Churn Rate for Security & Support Services
pivot_services = service_df.pivot(index='Service', columns='Subscription Status', values='Churn Rate (%)')

fig, ax = plt.subplots(figsize=(12, 6))
pivot_services[['No', 'Yes', 'No internet service']].plot(kind='bar', ax=ax, colormap='Spectral', edgecolor='black', width=0.7)
ax.set_title("Churn Rate by Service Subscription Status", fontsize=14, fontweight='bold')
ax.set_ylabel("Churn Rate (%)", fontsize=12)
ax.set_xlabel("Add-on Service", fontsize=12)
ax.set_ylim(0, 50)
ax.legend(title="Subscribed?", loc='upper right')

fig5_path = FIGURES_DIR / "02_services_churn_comparison.png"
plt.savefig(fig5_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {fig5_path.resolve()}")


--- Value-Added Services Churn Summary ---
         Service Subscription Status  Customer Count  Churn Rate (%)
  OnlineSecurity                  No            3498            41.8
  OnlineSecurity No internet service            1526             7.4
  OnlineSecurity                 Yes            2019            14.6
    OnlineBackup                  No            3088            39.9
    OnlineBackup No internet service            1526             7.4
    OnlineBackup                 Yes            2429            21.5
DeviceProtection                  No            3095            39.1
DeviceProtection No internet service            1526             7.4
DeviceProtection                 Yes            2422            22.5
     TechSupport                  No            3473            41.6
     TechSupport No internet service            1526             7.4
     TechSupport                 Yes            2044            15.2
     StreamingTV                  No            2810        

## 6. Demographic Profiling: Gender, Senior Citizens, Partners & Dependents
Finally, we evaluate whether customer demographic attributes associate with retention patterns.


In [7]:
demographics = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']

demo_records = []
for d in demographics:
    grouped = df.groupby(d)['Churn'].agg(
        total='count',
        churn_rate=lambda x: (x == 'Yes').mean() * 100
    )
    for category in grouped.index:
        label = "Senior (1)" if (d == 'SeniorCitizen' and category == 1) else ("Non-Senior (0)" if d == 'SeniorCitizen' else str(category))
        demo_records.append({
            "Demographic Attribute": d,
            "Subgroup": label,
            "Total Accounts": grouped.loc[category, 'total'],
            "Churn Rate (%)": round(grouped.loc[category, 'churn_rate'], 2)
        })

demo_df = pd.DataFrame(demo_records)
print("--- Demographic Churn Profile ---")
print(demo_df.to_string(index=False))


--- Demographic Churn Profile ---
Demographic Attribute       Subgroup  Total Accounts  Churn Rate (%)
               gender         Female            3488           26.92
               gender           Male            3555           26.16
        SeniorCitizen Non-Senior (0)            5901           23.61
        SeniorCitizen     Senior (1)            1142           41.68
              Partner             No            3641           32.96
              Partner            Yes            3402           19.66
           Dependents             No            4933           31.28
           Dependents            Yes            2110           15.45


## 7. Key Findings & Synthesis
1. **Contract Type Dominance:** Month-to-month contracts experience a staggering **42.7% churn rate**, compared to **11.3%** for 1-year and **2.8%** for 2-year contracts.
2. **Early-Tenure Hazard:** Customers in their first 6 months exhibit a **53.0% churn rate**. By month 24, churn stabilizes below 20%.
3. **Fiber Optic Attrition Paradox:** Fiber optic customers churn at **41.9%** (vs. 19.0% for DSL), driven by premium charges without bundled tech support.
4. **Electronic Check Disadvantage:** Customers paying by Electronic Check churn at **45.3%**, more than double automated credit card/bank transfer payers (~15.2%–16.7%).
5. **Support as a Churn Shield:** Customers with `TechSupport` and `OnlineSecurity` churn at ~14.6%–15.2%, compared to ~41.6%–41.8% for customers without these services.
6. **Gender Neutrality:** Gender has zero material relationship to churn (Male: 26.2% vs. Female: 26.9%).
